### **Setup**

In [ ]:
!rm -rf drone_detection
from kaggle_secrets import UserSecretsClient
gh_token = UserSecretsClient().get_secret("GH_TOKEN")
!git clone https://{gh_token}@github.com/Hannysia/drone_detection.git

In [ ]:
for pkg in ("numpy", "scipy"):
    !rm -rf /usr/local/lib/python3.12/dist-packages/{pkg}
    !rm -rf /usr/local/lib/python3.12/dist-packages/{pkg}.libs
    !rm -rf /usr/local/lib/python3.12/dist-packages/{pkg}-*.dist-info

!pip install -r drone_detection/requirements-kaggle.txt -q
!pip install -e drone_detection/ -q
%cd /kaggle/working/drone_detection

### **Tracker Search Spaces & Data Preparation**

In [ ]:
import itertools
import pandas as pd
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi
from IPython.display import display

from src.config import (
    HF_TOKEN, REPO_ID, LOCAL_HF_DATA_DIR, MOT_DATASET_DIR, SEQMAPS_DIR, WORKING_DIR, 
    YOLO_DETECTIONS_OUT, YOLO_MODEL_PATH_PT, TRACKERS_OUT_DIR, VIDEO_EXTENSIONS
)
from src.processing.dataset import compile_kaggle_mot_dataset
from src.processing.tracking import run_tracking_pipeline, run_yolo_inference_for_tracking
from src.processing.evaluation import run_trackeval, parse_trackeval_summary, find_and_display_best_configs

BYTETRACK_GRID = {
    "track_thresh": [0.3, 0.4, 0.5, 0.6, 0.7],
    "track_buffer": [15, 30, 45, 60],
    "match_thresh": [0.7, 0.8, 0.9]
}

OCSORT_GRID = {
    "det_thresh": [0.3, 0.4, 0.5, 0.6, 0.7],
    "max_age": [15, 30, 45, 60],
    "iou_threshold": [0.2, 0.3, 0.4, 0.5]
}

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [ ]:
print("Downloading required folders from Hugging Face...")
LOCAL_HF_DATA_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN,
    local_dir=str(LOCAL_HF_DATA_DIR),
    allow_patterns=["test/**", "val/**", "labels/**", "models/yolo26n_final_model_best.pt"]
)
print("Download from Hugging Face complete!")

In [ ]:
compile_kaggle_mot_dataset(LOCAL_HF_DATA_DIR, MOT_DATASET_DIR, SEQMAPS_DIR, tuple(VIDEO_EXTENSIONS))

### **Detector Inference**

In [ ]:
run_yolo_inference_for_tracking(YOLO_MODEL_PATH_PT, SEQMAPS_DIR, LOCAL_HF_DATA_DIR, YOLO_DETECTIONS_OUT, tuple(VIDEO_EXTENSIONS))

In [ ]:
# 1. Upload to HF
api = HfApi(token=HF_TOKEN)
api.upload_folder(folder_path=str(YOLO_DETECTIONS_OUT), path_in_repo="yolo_detections", repo_id=REPO_ID, repo_type="dataset")

# 2. Download back from HF
YOLO_VAL_DETECTIONS = WORKING_DIR / "yolo_detections" / "val"
YOLO_VAL_DETECTIONS.parent.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN, local_dir=str(WORKING_DIR), allow_patterns=["yolo_detections/**"])

### **Tracker Calibration (Validation Split)**

In [ ]:
# --- 1. Run ByteTrack Grid ---

bytetrack_runs = []
keys, values = zip(*BYTETRACK_GRID.items())
bytetrack_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

bytetrack_runs_model = []
print("Starting ByteTrack Grid Search on YOLO26 detections...")
for idx, config in enumerate(bytetrack_combinations):
    run_name = f"bytetrack_model_run_{idx}"
    bytetrack_runs_model.append(run_name)
    run_tracking_pipeline(
        split_name="val", tracker_name="bytetrack", tracker_config=config, 
        run_dir_name=run_name, dataset_base_dir=MOT_DATASET_DIR, trackers_out_dir=TRACKERS_OUT_DIR,
        detections_source_dir=YOLO_VAL_DETECTIONS
    )

run_trackeval(split_name="val", trackers_to_eval=bytetrack_runs_model, dataset_base_dir=MOT_DATASET_DIR, trackers_out_dir=TRACKERS_OUT_DIR)

In [ ]:
# --- 2. Run OC-SORT Grid ---

ocsort_runs = []
keys, values = zip(*OCSORT_GRID.items())
ocsort_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

ocsort_runs_model = []
print("\nStarting OC-SORT Grid Search on YOLO26 detections...")
for idx, config in enumerate(ocsort_combinations):
    run_name = f"ocsort_model_run_{idx}"
    ocsort_runs_model.append(run_name)
    run_tracking_pipeline(
        split_name="val", tracker_name="ocsort", tracker_config=config, 
        run_dir_name=run_name, dataset_base_dir=MOT_DATASET_DIR, trackers_out_dir=TRACKERS_OUT_DIR,
        detections_source_dir=YOLO_VAL_DETECTIONS
    )

run_trackeval(split_name="val", trackers_to_eval=ocsort_runs_model, dataset_base_dir=MOT_DATASET_DIR, trackers_out_dir=TRACKERS_OUT_DIR)

In [24]:
best_bytetrack_model, best_ocsort_model = find_and_display_best_configs(
    trackers_out_dir=TRACKERS_OUT_DIR,
    bytetrack_combinations=bytetrack_combinations,
    ocsort_combinations=ocsort_combinations,
    split_name="val",
    suffix="_model"
)


                 BEST CALIBRATED CONFIGURATIONS ON VAL SPLIT


Tracker,HOTA,IDF1,MOTA,FP,FN,IDSW
ByteTrack,68.97%,84.97%,76.41%,566,1272,27
OC-SORT,60.84%,69.88%,67.38%,382,2124,73


### **Final Evaluation on Test Split (YOLO + Trackers)**

In [25]:
YOLO_TEST_DETECTIONS = WORKING_DIR / "yolo_detections" / "test"

# 1. Final deployment of ByteTrack on YOLO test datasets
run_tracking_pipeline(
    split_name="test", 
    tracker_name="bytetrack", 
    tracker_config=best_bytetrack_model, 
    run_dir_name="bytetrack_yolo_final",
    detections_source_dir=YOLO_TEST_DETECTIONS,
    dataset_base_dir=MOT_DATASET_DIR,
    trackers_out_dir=TRACKERS_OUT_DIR
)

# 2. Final run of OC-SORT on YOLO test detections
run_tracking_pipeline(
    split_name="test", 
    tracker_name="ocsort", 
    tracker_config=best_ocsort_model, 
    run_dir_name="ocsort_yolo_final",
    detections_source_dir=YOLO_TEST_DETECTIONS,
    dataset_base_dir=MOT_DATASET_DIR,
    trackers_out_dir=TRACKERS_OUT_DIR
)

run_trackeval(
    split_name="test", 
    trackers_to_eval=["bytetrack_yolo_final", "ocsort_yolo_final"],
    dataset_base_dir=MOT_DATASET_DIR,
    trackers_out_dir=TRACKERS_OUT_DIR
)

===> [BYTETRACK] BYTETRACK_YOLO_FINAL | Config: {'track_thresh': 0.7, 'track_buffer': 60, 'match_thresh': 0.9} | Avg Latency: 0.645 ms/frame | Avg FPS: 1549.5
===> [OCSORT] OCSORT_YOLO_FINAL | Config: {'det_thresh': 0.5, 'max_age': 15, 'iou_threshold': 0.2} | Avg Latency: 0.784 ms/frame | Avg FPS: 1275.5

CLEAR Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          
Evaluating 2 configurations on split 'test'...

Evaluating 2 tracker(s) on 84 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating bytetrack_yolo_final


Evaluating ocsort_yolo_final

Evaluation complete. Results saved to disk successfully.


In [26]:
best_bytetrack_model, best_ocsort_model = find_and_display_best_configs(
    trackers_out_dir=TRACKERS_OUT_DIR,
    bytetrack_combinations=bytetrack_combinations,
    ocsort_combinations=ocsort_combinations,
    split_name="test",
    suffix="_model"
)

In [28]:
bt_test_yolo_metrics = parse_trackeval_summary(
    TRACKERS_OUT_DIR / "drone-test" / "bytetrack_yolo_final" / "pedestrian_summary.txt"
)
oc_test_yolo_metrics = parse_trackeval_summary(
    TRACKERS_OUT_DIR / "drone-test" / "ocsort_yolo_final" / "pedestrian_summary.txt"
)

df_test_yolo = pd.DataFrame(
    [bt_test_yolo_metrics, oc_test_yolo_metrics],
    index=["ByteTrack (YOLO Test)", "OC-SORT (YOLO Test)"]
)

display(df_test_yolo)

,HOTA,DetA,AssA,DetRe,DetPr,AssRe,AssPr,LocA,OWTA,HOTA(0),LocA(0),HOTALocA(0),MOTA,MOTP,MODA,CLR_Re,CLR_Pr,MTR,PTR,MLR,CLR_TP,CLR_FN,CLR_FP,IDSW,MT,PT,ML,Frag,sMOTA,IDF1,IDR,IDP,IDTP,IDFN,IDFP,Dets,GT_Dets,IDs,GT_IDs
ByteTrack (YOLO Test),62.564,60.031,65.625,66.969,77.945,70.014,85.335,85.502,66.225,75.583,82.609,62.438,69.595,83.407,69.902,77.910,90.679,67.059,24.706,8.2353,6606.0,1873.0,679.0,26.0,57.0,21.0,7.0,37.0,56.668,77.912,72.426,84.296,6141.0,2338.0,1144.0,7285.0,8479.0,136.0,85.0
OC-SORT (YOLO Test),56.657,57.465,56.027,61.162,87.159,57.662,92.441,91.019,58.520,61.852,89.827,55.560,61.175,90.125,62.036,66.104,94.202,38.824,49.412,11.7650,5605.0,2874.0,345.0,73.0,33.0,42.0,10.0,12.0,54.647,65.202,55.478,79.059,4704.0,3775.0,1246.0,5950.0,8479.0,175.0,85.0
